### Es 1
Hai a disposizione un file `data.csv` contenente dati mensili di traffico aereo con due colonne:

- `date`: data in formato `YYYY-MM` (mese/anno)
- `passengers`: numero di passeggeri per quel mese


Costruisci un modello di **regressione polinomiale** che approssima l’andamento del numero di passeggeri nel tempo.

1. Carica il dataset.
2. Convertilo in un formato numerico utilizzando una colonna `mese_numerico` che conti i mesi a partire da gennaio 1949.
3. Applica una regressione polinomiale (grado a tua scelta).
4. Calcola l’RMSE tra i valori reali e quelli predetti.
5. Visualizza i dati reali e la curva stimata con Plotly.

In [9]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
import plotly.graph_objects as go
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split



# 1. Caricamento del dataset


try:
    df = pd.read_csv('data.csv')
except FileNotFoundError:
    # Generiamo un dataset fittizio di esempio (AirPassengers-like) se il file non esiste
    date_range = pd.date_range(start='1949-01', end='1960-12', freq='MS')
    mesi_totali = len(date_range)
    # Una componente lineare + oscillazione sinusoidale + rumore casuale
    passengers_trend = 100 + 2.5 * np.arange(mesi_totali) + 30 * np.sin(np.arange(mesi_totali) * (2 * np.pi / 12))
    df = pd.DataFrame({
        'date': date_range.strftime('%Y-%m'),
        'passengers': passengers_trend.astype(int)
    })


# 2. Conversione in formato numerico (mese_numerico)

# Convertiamo la colonna date in oggetti datetime
df['date_dt'] = pd.to_datetime(df['date'], format='%Y-%m')

# Calcoliamo la differenza in mesi rispetto a Gennaio 1949

df['mese_numerico'] = (df['date_dt'].dt.year - 1949) * 12 + (df['date_dt'].dt.month - 1)

# Preparazione delle variabili per il modello
X = df[['mese_numerico']].values
y = df['passengers'].values

# 3. Applicazione della Regressione Polinomiale

# Scegliamo un grado 
grado_polinomio = 3

poly_features = PolynomialFeatures(degree=grado_polinomio)
X_poly = poly_features.fit_transform(X)

# Addestramento del modello lineare sulle feature polinomiali
model = LinearRegression()
model.fit(X_poly, y)

# Predizione dei valori
y_pred = model.predict(X_poly)

# 4. Calcolo dell’RMSE (Root Mean Squared Error)

RMSE = np.sqrt(np.mean(y - y_pred**2))
print(f"Grado del polinomio scelto: {grado_polinomio}")
print(f"RMSE : {RMSE:.2f}")
# 5. Visualizzazione con Plotly

fig = go.Figure()

# Aggiungiamo i dati reali (Scatter plot)
fig.add_trace(go.Scatter(
    x=df['date'], 
    y=y, 
    mode='markers+lines', 
    name='Dati Reali',
    line=dict(color='blue', width=1),
    marker=dict(size=4)
))

# Aggiungiamo la curva stimata dal modello polinomiale (Line plot)
fig.add_trace(go.Scatter(
    x=df['date'], 
    y=y_pred, 
    mode='lines', 
    name=f'Curva Polinomiale (Grado {grado_polinomio})',
    line=dict(color='red', width=3)
))

# Layout del grafico
fig.update_layout(
    title='Approssimazione del Traffico Aereo con Regressione Polinomiale',
    xaxis_title='Data (Mese-Anno)',
    yaxis_title='Numero di Passeggeri',
    hovermode='x unified',
    template='plotly_white',
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

# Mostra il grafico interattivo
fig.write_html("grafico.html")

Grado del polinomio scelto: 3
RMSE : nan


C:\Users\hp\AppData\Local\Temp\ipykernel_18596\1890686563.py:59: RuntimeWarning: invalid value encountered in sqrt
  RMSE = np.sqrt(np.mean(y - y_pred**2))
